# Promotion Trend Analysis

## Project Overview
Analyze promotion data to study how promotions vary by team, tenure, performance, and employee characteristics.

## Learning Objectives
- Calculate promotion rates across organizational dimensions
- Identify promotion velocity patterns by department and level
- Analyze the relationship between performance ratings and promotion outcomes
- Detect potential bias in promotion decisions

## Problem Statement
Leadership wants to ensure promotions are merit-based and equitable. They need data to understand promotion rates, timing, and whether certain groups are systematically over- or under-promoted relative to performance.

## Why This Project Matters
Promotion fairness directly impacts employee retention, engagement, and legal compliance. Understanding promotion patterns helps build a stronger, more equitable talent pipeline.

## Dataset Overview
Synthetic workforce dataset: ~3,500 employees with promotion history, performance, tenure, department, and demographic data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3500
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Finance', 'Operations', 'Product'],
                                 n, p=[0.25, 0.20, 0.15, 0.12, 0.16, 0.12])
current_level = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead', 'Manager'], n, p=[0.22, 0.30, 0.26, 0.14, 0.08])
gender = np.random.choice(['Male', 'Female', 'Non-Binary'], n, p=[0.51, 0.45, 0.04])
tenure_years = np.clip(np.random.exponential(4.5, n), 0.5, 20).round(1)
avg_perf = np.clip(np.random.normal(3.4, 0.7, n), 1.0, 5.0).round(1)
age = np.clip(np.random.normal(34, 7, n).astype(int), 22, 58)

# Promotion logic
promo_prob = 0.05 +     0.15 * (avg_perf >= 4.0) +     0.08 * (avg_perf >= 3.5) +     0.05 * (tenure_years >= 2) +     0.03 * (tenure_years >= 4) +     np.where(current_level == 'Junior', 0.05, 0) +     np.where(gender == 'Female', -0.02, 0) +     np.random.normal(0, 0.02, n)
promo_prob = np.clip(promo_prob, 0.02, 0.50)
promoted = (np.random.random(n) < promo_prob).astype(int)

months_since_last_promo = np.where(promoted, 0, np.random.randint(6, 48, n))
time_to_promote = np.where(promoted, np.clip(np.random.exponential(18, n).astype(int), 6, 60), 0)

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Department': departments, 'CurrentLevel': current_level,
    'Gender': gender, 'Age': age, 'TenureYears': tenure_years,
    'AvgPerfRating': avg_perf, 'PromotedThisYear': promoted,
    'MonthsSinceLastPromo': months_since_last_promo,
    'TimeToPromoteMonths': time_to_promote,
})
df['TenureBand'] = pd.cut(df['TenureYears'], bins=[0, 1, 2, 4, 8, 99], labels=['<1yr', '1-2yr', '2-4yr', '4-8yr', '8yr+'])
df['PerfBand'] = pd.cut(df['AvgPerfRating'], bins=[0, 2.5, 3.5, 4.5, 5.1], labels=['Low', 'Medium', 'High', 'Top'])

print(f'Dataset: {df.shape}')
print(f'Overall promotion rate: {df["PromotedThisYear"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nLevel distribution:\n{df["CurrentLevel"].value_counts()}')
print(f'\nPromotion counts: {df["PromotedThisYear"].sum()} / {len(df)}')

## Overall Promotion Rates by Department

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_promo = df.groupby('Department')['PromotedThisYear'].mean().sort_values()
dept_promo.plot.barh(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Promotion Rate by Department')
axes[0].set_xlabel('Promotion Rate')

level_promo = df.groupby('CurrentLevel')['PromotedThisYear'].mean().reindex(['Junior','Mid','Senior','Lead','Manager'])
level_promo.plot.bar(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Promotion Rate by Current Level')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'promo_dept_level.png'), dpi=100, bbox_inches='tight')
plt.show()

## Performance vs Promotion

In [ ]:
perf_promo = df.groupby('PerfBand').agg(
    employees=('EmployeeID', 'count'),
    promoted=('PromotedThisYear', 'sum'),
    promo_rate=('PromotedThisYear', 'mean'),
).round(3)
print('Promotion Rate by Performance Band:')
print(perf_promo)

fig, ax = plt.subplots(figsize=(8, 5))
perf_promo['promo_rate'].plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Promotion Rate by Performance Band')
ax.set_ylabel('Promotion Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'perf_promotion.png'), dpi=100, bbox_inches='tight')
plt.show()

## Gender Promotion Analysis

In [ ]:
gender_promo = df.groupby('Gender').agg(
    employees=('EmployeeID', 'count'),
    promo_rate=('PromotedThisYear', 'mean'),
    avg_perf=('AvgPerfRating', 'mean'),
    avg_tenure=('TenureYears', 'mean'),
).round(3)
print('Promotion by Gender:')
print(gender_promo)

# Controlled by performance
gender_perf = df.groupby(['PerfBand', 'Gender'])['PromotedThisYear'].mean().unstack().round(3)
print('\nPromotion Rate — Performance × Gender:')
print(gender_perf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gender_promo['promo_rate'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Raw Promotion Rate by Gender')
axes[0].tick_params(axis='x', rotation=0)

gender_perf.plot.bar(ax=axes[1], edgecolor='black')
axes[1].set_title('Promotion Rate: Performance × Gender')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gender_promotion.png'), dpi=100, bbox_inches='tight')
plt.show()

## Tenure Impact on Promotion

In [ ]:
tenure_promo = df.groupby('TenureBand').agg(
    employees=('EmployeeID', 'count'),
    promo_rate=('PromotedThisYear', 'mean'),
).round(3)
print('Promotion by Tenure Band:')
print(tenure_promo)

fig, ax = plt.subplots(figsize=(8, 5))
tenure_promo['promo_rate'].plot.bar(ax=ax, edgecolor='black', color='goldenrod')
ax.set_title('Promotion Rate by Tenure Band')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tenure_promotion.png'), dpi=100, bbox_inches='tight')
plt.show()

## Promotion Velocity — Time to Promote

In [ ]:
promoted_df = df[df['PromotedThisYear'] == 1]
print(f'Promoted employees: {len(promoted_df)}')
print(f'Time-to-promote stats (months):\n{promoted_df["TimeToPromoteMonths"].describe().round(1)}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(promoted_df['TimeToPromoteMonths'], bins=20, edgecolor='black', color='steelblue', alpha=0.7)
ax.axvline(promoted_df['TimeToPromoteMonths'].median(), color='red', linestyle='--',
           label=f'Median: {promoted_df["TimeToPromoteMonths"].median():.0f} months')
ax.set_title('Time-to-Promote Distribution')
ax.set_xlabel('Months')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'time_to_promote.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department × Level Promotion Heatmap

In [ ]:
cross = df.groupby(['CurrentLevel', 'Department'])['PromotedThisYear'].mean().unstack().round(3)
cross = cross.reindex(['Junior', 'Mid', 'Senior', 'Lead', 'Manager'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax)
ax.set_title('Promotion Rate: Level × Department')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'promo_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Performance is the strongest promotion driver**, as expected — Top performers promote at 3-4x the rate of Low performers
- **A small gender promotion gap exists** even when controlling for performance, suggesting potential bias in promotion decisions
- **Tenure correlates with promotion** up to a point — the 2-4 year band shows highest promotion rates (the "sweet spot")
- **Junior-to-Mid** transitions happen most frequently — Senior+ promotions are much rarer, creating a common career plateau
- **Department variation** is notable — some departments promote faster, likely reflecting different growth rates and headcount needs
- **Time-to-promote** averages ~18 months, with significant variance — some high performers wait much longer

## Limitations
- Synthetic data with a programmed gender gap
- No multi-year longitudinal tracking
- No qualitative factors (sponsorship, visibility, project impact)
- Single-year snapshot — no trend analysis
- Performance ratings are self-contained, no calibration process modeled

## How to Improve This Project
- Use multi-year data to track promotion velocity over time
- Add intersectional analysis (gender × race × level)
- Include skip-level promotions and lateral moves
- Model the effect of manager sponsorship on promotion outcomes
- Add compensation change at promotion time

## Production Considerations
- Annual promotion equity audit before and after calibration cycles
- Manager scorecards with team promotion rate vs org average
- Automated alerts for employees in promotion-eligible cohorts who haven't been promoted
- Integration with succession planning tools

## Common Mistakes
- Comparing raw promotion rates without controlling for performance and tenure
- Treating promotions as purely meritocratic without examining systemic patterns
- Small-sample comparisons leading to unreliable conclusions
- Ignoring the denominator problem — departments with more senior people have fewer promotion opportunities

## Mini Challenge / Exercises
1. Calculate the promotion rate for high-performing women vs high-performing men. Is there a gap after controlling for performance?
2. Which department has the fastest median time-to-promote?
3. What percentage of employees with 3+ years tenure and Top performance have NOT been promoted?
4. Build a simple promotion eligibility score using tenure, performance, and months since last promotion.

## Final Summary / Key Takeaways
- Promotion analysis reveals whether organizations advance talent fairly
- Performance should be the primary driver — but controlling for other factors reveals potential bias
- Tenure-based patterns help calibrate promotion timing expectations
- Department-level variation is expected but should be monitored for excessive disparity
- Regular, controlled promotion equity audits are essential for building trust and retaining talent